In [ ]:
# =============================================================================
# CELL 1: DATA LOADING & PREPROCESSING
# =============================================================================
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.linear_model import Ridge, Lasso, ElasticNet, HuberRegressor
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor, RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.feature_selection import mutual_info_regression
from sklearn.base import clone
from xgboost import XGBRegressor
import lightgbm as lgb
import optuna
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Load data
owid_df = pd.read_csv('../data/filtered_compact.csv')
owid_df['date'] = pd.to_datetime(owid_df['date'])
de = owid_df[owid_df['country'] == 'Germany'].copy()
de = de.sort_values('date').reset_index(drop=True)

# Only use data up to mid-2022
de = de[de['date'] <= '2022-06-30'].copy()

# Forward-fill within the ACTIVE reporting period only
key_cols = ['new_deaths', 'new_deaths_smoothed', 'new_cases', 'new_cases_smoothed',
            'reproduction_rate', 'icu_patients', 'positive_rate',
            'weekly_hosp_admissions', 'stringency_index', 'new_tests_smoothed',
            'people_fully_vaccinated']
for col in key_cols:
    if col in de.columns:
        de[col] = de[col].ffill().bfill()

if 'new_deaths_smoothed' not in de.columns or de['new_deaths_smoothed'].isna().all():
    de['new_deaths_smoothed'] = de['new_deaths'].rolling(14, min_periods=1).mean()
if 'new_cases_smoothed' not in de.columns or de['new_cases_smoothed'].isna().all():
    de['new_cases_smoothed'] = de['new_cases'].rolling(14, min_periods=1).mean()

print(f"Germany active data: {len(de)} rows, {de['date'].min().date()} to {de['date'].max().date()}")

In [ ]:
# =============================================================================
# CELL 2: FEATURE ENGINEERING
# =============================================================================
def engineer_features_v3(df, target_type='deaths'):
    data = df.copy()
    data = data.sort_values('date').reset_index(drop=True)

    if target_type == 'deaths':
        data['target'] = data['new_deaths_smoothed'].clip(lower=0)
        target_name = 'Daily Deaths (smoothed)'
    else:
        data['target'] = data['new_cases_smoothed'].clip(lower=0)
        target_name = '14-day Smoothed Cases'

    features = []
    MIN_LAG = 14

    # GROUP 1: Core autoregressive (3 features)
    for lag in [14, 21, 28]:
        fname = f'lag_{lag}'
        data[fname] = data['target'].shift(lag)
        features.append(fname)

    # GROUP 2: Log-space lags
    for lag in [14, 21, 28]:
        fname = f'log_lag_{lag}'
        data[fname] = np.log1p(data['target'].shift(lag))
        features.append(fname)

    # GROUP 3: Scale-invariant growth rates
    for period in [14, 21, 28]:
        denom = data['target'].shift(MIN_LAG + period).replace(0, np.nan)
        numer = data['target'].shift(MIN_LAG)
        fname = f'growth_ratio_{period}d'
        data[fname] = (numer / denom).clip(0.05, 20)
        features.append(fname)
        fname_log = f'log_growth_{period}d'
        data[fname_log] = (np.log1p(numer) - np.log1p(denom)).clip(-3, 3)
        features.append(fname_log)

    # GROUP 4: Acceleration
    growth_14 = data['target'].shift(MIN_LAG) / data['target'].shift(MIN_LAG + 21).replace(0, np.nan)
    growth_14_prev = data['target'].shift(MIN_LAG + 21) / data['target'].shift(MIN_LAG + 21).replace(0, np.nan)
    data['acceleration'] = (growth_14 / growth_14_prev.replace(0, np.nan)).clip(0.1, 10)
    features.append('acceleration')

    # GROUP 5: Normalized trend slope
    shifted = data['target'].shift(MIN_LAG)
    for lookback in [14, 21]:
        slopes = []
        for i in range(len(data)):
            start = i - MIN_LAG - lookback + 1
            end = i - MIN_LAG + 1
            if start >= 0:
                y_vals = data['target'].iloc[start:end].values
                if len(y_vals) == lookback and np.all(np.isfinite(y_vals)):
                    x = np.arange(lookback, dtype=float)
                    try:
                        slope = np.polyfit(x, y_vals, 1)[0]
                        slopes.append(slope)
                    except:
                        slopes.append(np.nan)
                else:
                    slopes.append(np.nan)
            else:
                slopes.append(np.nan)
        data[f'slope_{lookback}d'] = slopes
        mean_val = shifted.rolling(lookback, min_periods=1).mean().replace(0, np.nan)
        data[f'norm_slope_{lookback}d'] = (data[f'slope_{lookback}d'] / mean_val).clip(-2, 2)
        features.append(f'norm_slope_{lookback}d')
        features.append(f'slope_{lookback}d')

    # GROUP 6: Volatility (coefficient of variation)
    for window in [14, 21]:
        roll_mean = shifted.rolling(window, min_periods=3).mean()
        roll_std = shifted.rolling(window, min_periods=3).std()
        fname = f'cv_{window}d'
        data[fname] = (roll_std / roll_mean.replace(0, np.nan)).clip(0, 3).fillna(0)
        features.append(fname)

    # GROUP 7: Relative position in recent range
    roll_min_14 = shifted.rolling(14, min_periods=3).min()
    roll_max_14 = shifted.rolling(14, min_periods=3).max()
    roll_range = (roll_max_14 - roll_min_14).replace(0, np.nan)
    data['range_position'] = ((shifted - roll_min_14) / roll_range).fillna(0.5)
    features.append('range_position')

    # GROUP 8: Key external indicators (lagged)
    external_map = {
        'reproduction_rate': 'Rt',
        'icu_patients': 'icu',
        'positive_rate': 'posrate',
        'stringency_index': 'stringency',
    }
    for col, short in external_map.items():
        if col in data.columns and data[col].notna().sum() > 100:
            data[f'{short}_lag14'] = data[col].shift(14)
            features.append(f'{short}_lag14')
            denom_ext = data[col].shift(14).replace(0, np.nan)
            data[f'{short}_growth'] = (data[col].shift(14) / denom_ext).clip(0.1, 10)
            features.append(f'{short}_growth')

    # GROUP 9: Cross-target features
    if target_type == 'deaths' and 'new_cases_smoothed' in data.columns:
        data['cases_lag14'] = data['new_cases_smoothed'].shift(14)
        data['cases_lag21'] = data['new_cases_smoothed'].shift(21)
        data['log_cases_lag14'] = np.log1p(data['new_cases_smoothed'].shift(14))
        case_denom = data['new_cases_smoothed'].shift(14).replace(0, np.nan)
        data['cfr_approx'] = (data['target'].shift(14) / case_denom).clip(0, 0.2)
        features.extend(['cases_lag14', 'log_cases_lag14', 'cfr_approx'])
    elif target_type == 'cases' and 'new_deaths_smoothed' in data.columns:
        data['deaths_lag14'] = data['new_deaths_smoothed'].shift(14)
        data['log_deaths_lag14'] = np.log1p(data['new_deaths_smoothed'].shift(14))
        features.extend(['deaths_lag14', 'log_deaths_lag14'])

    # GROUP 10: Seasonality
    data['sin_weekly'] = np.sin(2 * np.pi * data['date'].dt.dayofweek / 7)
    data['cos_weekly'] = np.cos(2 * np.pi * data['date'].dt.dayofweek / 7)
    data['sin_yearly'] = np.sin(2 * np.pi * data['date'].dt.dayofyear / 365.25)
    data['cos_yearly'] = np.cos(2 * np.pi * data['date'].dt.dayofyear / 365.25)
    features.extend(['sin_weekly', 'cos_weekly', 'sin_yearly', 'cos_yearly'])

    # CLEANUP
    features = [f for f in features if f in data.columns]
    keep = ['date', 'target'] + features
    data = data[keep]
    data = data.replace([np.inf, -np.inf], np.nan)
    for f in features:
        data[f] = data[f].fillna(0)
    data = data.dropna(subset=['target'])
    data = data[data['date'] >= '2020-03-15']

    print(f"  {target_name}: {len(data)} rows, {len(features)} features")
    print(f"  Date range: {data['date'].min().date()} to {data['date'].max().date()}")
    return data, features, target_name

print("=" * 70)
print("FEATURE ENGINEERING")
print("=" * 70)
de_deaths, feats_d, tname_d = engineer_features_v3(de, 'deaths')
de_cases, feats_c, tname_c = engineer_features_v3(de, 'cases')

In [ ]:
# =============================================================================
# CELL 3: TEMPORAL SPLIT
# =============================================================================
def temporal_split_v3(data, label=''):
    train = data[(data['date'] >= '2020-03-15') & (data['date'] <= '2021-12-31')].copy()
    val   = data[(data['date'] >= '2021-01-01') & (data['date'] < '2021-02-01')].copy()
    test  = data[(data['date'] >= '2022-01-01') & (data['date'] < '2022-02-01')].copy()

    assert train['date'].max() < pd.Timestamp('2022-01-01'), "LEAKAGE!"

    print(f"\n  {label}:")
    for name, s in [('Train', train), ('Val', val), ('Test', test)]:
        if len(s) > 0:
            print(f"    {name}: {len(s)} rows | "
                  f"target mean={s['target'].mean():.1f}, "
                  f"std={s['target'].std():.1f}, "
                  f"range=[{s['target'].min():.1f}, {s['target'].max():.1f}]")

    train_max = train['target'].max()
    test_max = test['target'].max() if len(test) > 0 else 0
    if test_max > train_max * 1.2:
        print(f"    OOD WARNING: Test max ({test_max:.0f}) >> Train max ({train_max:.0f})")
    elif len(test) > 0 and test['target'].mean() < train['target'].quantile(0.25):
        print(f"    OOD WARNING: Test mean ({test['target'].mean():.0f}) << Train Q25 ({train['target'].quantile(0.25):.0f})")

    return {'train': train, 'val': val, 'test': test}

print("=" * 70)
print("TEMPORAL SPLIT")
print("=" * 70)
split_d = temporal_split_v3(de_deaths, 'Deaths')
split_c = temporal_split_v3(de_cases, 'Cases')

In [ ]:
# =============================================================================
# CELL 4: FEATURE SELECTION
# =============================================================================
def select_features_v3(train, features, max_features=15, corr_thresh=0.90):
    X = train[features]
    y = train['target']

    mi_scores = mutual_info_regression(X, y, random_state=42, n_neighbors=5)
    mi_ranking = pd.Series(mi_scores, index=features).sort_values(ascending=False)
    lin_corr = X.corrwith(y).abs().sort_values(ascending=False)

    mi_ranks = mi_ranking.rank(ascending=False)
    lin_ranks = lin_corr.reindex(features).rank(ascending=False)
    combined = (mi_ranks * lin_ranks) ** 0.5
    combined = combined.sort_values()

    selected = []
    scale_invariant = {'growth_ratio', 'log_growth', 'norm_slope', 'acceleration',
                       'cv_', 'range_position', 'cfr_approx', '_growth'}

    for feat in combined.index:
        if len(selected) >= max_features:
            break
        redundant = False
        for sel in selected:
            if abs(X[feat].corr(X[sel])) > corr_thresh:
                redundant = True
                break
        if not redundant:
            selected.append(feat)

    si_count = sum(1 for f in selected if any(si in f for si in scale_invariant))
    if si_count < 3:
        for feat in combined.index:
            if feat not in selected and any(si in feat for si in scale_invariant):
                non_si = [f for f in selected if not any(si in f for si in scale_invariant)]
                if non_si:
                    selected.remove(non_si[-1])
                    selected.append(feat)
                    si_count += 1
                    if si_count >= 3:
                        break

    print(f"  Selected {len(selected)}/{len(features)} features")
    print(f"  Scale-invariant count: {si_count}")
    for f in selected:
        mi = mi_ranking.get(f, 0)
        lc = lin_corr.get(f, 0)
        print(f"    {f:<30} MI={mi:.3f}  |r|={lc:.3f}")
    return selected

print("=" * 70)
print("FEATURE SELECTION")
print("=" * 70)
print("\nDeaths:")
sel_d = select_features_v3(split_d['train'], feats_d)
print("\nCases:")
sel_c = select_features_v3(split_c['train'], feats_c)

In [ ]:
'takes 5 min 10 sec to run approx'


# =============================================================================
# CELL 5B: MODEL TRAINING — DEATHS ( SYSTEMATIC)
# =============================================================================
print("=" * 70)
print("MODEL TRAINING — GERMANY DEATHS ")
print("=" * 70)

import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.linear_model import ElasticNet, Lasso, Ridge, BayesianRidge, HuberRegressor
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.ensemble import GradientBoostingRegressor

train_d = split_d['train']
val_d   = split_d['val']
test_d  = split_d['test']
train_val_d = pd.concat([train_d, val_d], ignore_index=True)

results_d = {}

# Feature sets
all_d_feats = [f for f in feats_d if f in train_d.columns]

print(f"\n  Train: n={len(train_d)}, max={train_d['target'].max():.0f}, mean={train_d['target'].mean():.0f}")
print(f"  Val:   n={len(val_d)}, max={val_d['target'].max():.0f}, mean={val_d['target'].mean():.0f}")
print(f"  Test:  n={len(test_d)}, max={test_d['target'].max():.0f}, mean={test_d['target'].mean():.0f}")
print(f"  OOD ratio: {test_d['target'].mean()/train_d['target'].mean():.2f}x mean, "
      f"{test_d['target'].max()/train_d['target'].max():.2f}x max")
print(f"  Features: sel={len(sel_d)}, all={len(all_d_feats)}")

def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred) if len(y_true) > 1 else np.nan
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true < 1, 1, y_true))) * 100
    print(f"    {name}: MAE={mae:.2f}, RMSE={rmse:.2f}, R²={r2:.3f}, MAPE={mape:.1f}%")
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'mape': mape}

# ==========================================================================
# Scalers
# ==========================================================================
scaler_t_sel = RobustScaler()
X_t_sel = scaler_t_sel.fit_transform(train_d[sel_d])

scaler_tv_sel = RobustScaler()
X_tv_sel = scaler_tv_sel.fit_transform(train_val_d[sel_d])

scaler_t_full = RobustScaler()
X_t_full = scaler_t_full.fit_transform(train_d[all_d_feats])

scaler_tv_full = RobustScaler()
X_tv_full = scaler_tv_full.fit_transform(train_val_d[all_d_feats])

scaler_std_t = StandardScaler()
X_t_std = scaler_std_t.fit_transform(train_d[all_d_feats])

scaler_std_tv = StandardScaler()
X_tv_std = scaler_std_tv.fit_transform(train_val_d[all_d_feats])

y_t = train_d['target'].values
y_tv = train_val_d['target'].values
y_t_log = np.log1p(train_d['target']).values
y_tv_log = np.log1p(train_val_d['target']).values

# ==========================================================================
# ORIGINAL MODELS (reproduced here for standalone)
# ==========================================================================
print(f"\n{'='*60}")
print("  ORIGINAL MODELS (from Cell 5)")
print(f"{'='*60}")

# ---- XGBoost ----
print("\n  [1] XGBoost:")
xgb_d = XGBRegressor(
    n_estimators=500, max_depth=3, learning_rate=0.02,
    subsample=0.65, colsample_bytree=0.6,
    reg_alpha=1.0, reg_lambda=5.0, min_child_weight=15,
    gamma=0.5, random_state=42, n_jobs=-1, early_stopping_rounds=50
)
xgb_d.fit(train_d[sel_d], train_d['target'],
          eval_set=[(val_d[sel_d], val_d['target'])], verbose=False)

results_d['XGBoost'] = {}
for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
    pred = np.clip(xgb_d.predict(s[sel_d]), 0, None)
    results_d['XGBoost'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_d['XGBoost'][name.lower()].update(evaluate(f'XGB {name}', s['target'].values, pred))

# ---- XGBoost log-target ----
print("\n  [2] XGBoost (log-target):")
xgb_log_d = XGBRegressor(
    n_estimators=500, max_depth=3, learning_rate=0.02,
    subsample=0.65, colsample_bytree=0.6,
    reg_alpha=1.0, reg_lambda=5.0, min_child_weight=15,
    gamma=0.5, random_state=42, n_jobs=-1, early_stopping_rounds=50
)
xgb_log_d.fit(train_d[sel_d], np.log1p(train_d['target']),
              eval_set=[(val_d[sel_d], np.log1p(val_d['target']))], verbose=False)

results_d['XGBoost-Log'] = {}
for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
    pred = np.clip(np.expm1(xgb_log_d.predict(s[sel_d])), 0, None)
    results_d['XGBoost-Log'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_d['XGBoost-Log'][name.lower()].update(evaluate(f'XGB-Log {name}', s['target'].values, pred))

# ---- LightGBM MAE ----
print("\n  [3] LightGBM (MAE):")
lgb_d = lgb.LGBMRegressor(
    n_estimators=500, max_depth=3, learning_rate=0.02,
    subsample=0.65, colsample_bytree=0.6,
    reg_alpha=1.0, reg_lambda=5.0, min_child_samples=15,
    objective='mae', random_state=42, n_jobs=-1, verbose=-1
)
lgb_d.fit(train_d[sel_d], train_d['target'],
          eval_set=[(val_d[sel_d], val_d['target'])],
          callbacks=[lgb.early_stopping(50, verbose=False)])

results_d['LightGBM-MAE'] = {}
for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
    pred = np.clip(lgb_d.predict(s[sel_d]), 0, None)
    results_d['LightGBM-MAE'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_d['LightGBM-MAE'][name.lower()].update(evaluate(f'LGB-MAE {name}', s['target'].values, pred))

# ---- LightGBM Huber log-target ----
print("\n  [4] LightGBM (Huber, log-target):")
lgb_huber_d = lgb.LGBMRegressor(
    n_estimators=500, max_depth=3, learning_rate=0.02,
    subsample=0.65, colsample_bytree=0.6,
    reg_alpha=1.0, reg_lambda=5.0, min_child_samples=15,
    objective='huber', huber_delta=1.0,
    random_state=42, n_jobs=-1, verbose=-1
)
lgb_huber_d.fit(train_d[sel_d], np.log1p(train_d['target']),
                eval_set=[(val_d[sel_d], np.log1p(val_d['target']))],
                callbacks=[lgb.early_stopping(50, verbose=False)])

results_d['LightGBM-Huber-Log'] = {}
for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
    pred = np.clip(np.expm1(lgb_huber_d.predict(s[sel_d])), 0, None)
    results_d['LightGBM-Huber-Log'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_d['LightGBM-Huber-Log'][name.lower()].update(
        evaluate(f'LGB-Hub-Log {name}', s['target'].values, pred))

# ---- GB Huber ----
print("\n  [5] GradientBoosting (Huber):")
gb_d = GradientBoostingRegressor(
    n_estimators=300, max_depth=3, learning_rate=0.03,
    subsample=0.7, min_samples_leaf=15,
    loss='huber', alpha=0.9, random_state=42
)
gb_d.fit(train_d[sel_d], train_d['target'])

results_d['GB-Huber'] = {}
for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
    pred = np.clip(gb_d.predict(s[sel_d]), 0, None)
    results_d['GB-Huber'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_d['GB-Huber'][name.lower()].update(evaluate(f'GB {name}', s['target'].values, pred))

# ---- Ridge (selected features) ----
print("\n  [6] Ridge (selected features):")
ridge_d = Ridge(alpha=5.0)
ridge_d.fit(X_t_sel, train_d['target'])

results_d['Ridge'] = {}
for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
    X_sc = scaler_t_sel.transform(s[sel_d])
    pred = np.clip(ridge_d.predict(X_sc), 0, None)
    results_d['Ridge'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_d['Ridge'][name.lower()].update(evaluate(f'Ridge {name}', s['target'].values, pred))

# ---- Ridge log-target (selected features) ----
print("\n  [7] Ridge (log-target, selected):")
ridge_log_d = Ridge(alpha=5.0)
ridge_log_d.fit(X_t_sel, np.log1p(train_d['target']))

results_d['Ridge-Log'] = {}
for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
    X_sc = scaler_t_sel.transform(s[sel_d])
    pred = np.clip(np.expm1(ridge_log_d.predict(X_sc)), 0, None)
    results_d['Ridge-Log'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_d['Ridge-Log'][name.lower()].update(evaluate(f'Ridge-Log {name}', s['target'].values, pred))

# ---- Huber (Optuna-tuned, selected features) ----
print("\n  [8] Huber (Optuna, selected features, 100 trials):")
def objective_huber_sel(trial):
    h_alpha = trial.suggest_float("alpha", 1e-5, 10.0, log=True)
    h_epsilon = trial.suggest_float("epsilon", 1.1, 2.0)
    model = HuberRegressor(alpha=h_alpha, epsilon=h_epsilon, max_iter=2000)
    model.fit(X_t_sel, train_d['target'])
    preds = model.predict(scaler_t_sel.transform(val_d[sel_d]))
    return mean_absolute_error(val_d['target'], preds)

study_hub = optuna.create_study(direction="minimize")
study_hub.optimize(objective_huber_sel, n_trials=100)
print(f"    Best: {study_hub.best_params}, Val MAE={study_hub.best_value:.2f}")

huber_d = HuberRegressor(**study_hub.best_params, max_iter=2000)
huber_d.fit(X_t_sel, train_d['target'])

results_d['Huber'] = {}
for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
    X_sc = scaler_t_sel.transform(s[sel_d])
    pred = np.clip(huber_d.predict(X_sc), 0, None)
    results_d['Huber'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_d['Huber'][name.lower()].update(evaluate(f'Huber {name}', s['target'].values, pred))

# ==========================================================================
# V8 NEW: SYSTEMATIC LASSO SWEEP (log-target, full features)
# ==========================================================================
print(f"\n{'='*60}")
print("  V8 NEW: LASSO ALPHA SWEEP (log-target, full features)")
print(f"{'='*60}")

print("\n  [9] Lasso sweep (RobustScaler, train-only):")
for alpha in [0.001, 0.003, 0.005, 0.007, 0.01, 0.015, 0.02, 0.03, 0.05, 0.1]:
    model = Lasso(alpha=alpha, max_iter=20000, tol=1e-6)
    model.fit(X_t_full, y_t_log)
    
    mname = f'Lasso_{alpha}_T'
    results_d[mname] = {}
    for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
        X = scaler_t_full.transform(s[all_d_feats])
        pred = np.clip(np.expm1(model.predict(X)), 0, None)
        results_d[mname][name.lower()] = {
            'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
        }
        results_d[mname][name.lower()].update(
            evaluate(f'{mname} {name}', s['target'].values, pred))

print("\n  [10] Lasso sweep (RobustScaler, train+val):")
for alpha in [0.001, 0.003, 0.005, 0.007, 0.01, 0.015, 0.02, 0.03, 0.05, 0.1]:
    model = Lasso(alpha=alpha, max_iter=20000, tol=1e-6)
    model.fit(X_tv_full, y_tv_log)
    
    mname = f'Lasso_{alpha}_TV'
    results_d[mname] = {}
    for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
        X = scaler_tv_full.transform(s[all_d_feats])
        pred = np.clip(np.expm1(model.predict(X)), 0, None)
        results_d[mname][name.lower()] = {
            'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
        }
        results_d[mname][name.lower()].update(
            evaluate(f'{mname} {name}', s['target'].values, pred))

print("\n  [11] Lasso sweep (StandardScaler, train-only):")
for alpha in [0.001, 0.003, 0.005, 0.007, 0.01, 0.015, 0.02, 0.03, 0.05, 0.1]:
    model = Lasso(alpha=alpha, max_iter=20000, tol=1e-6)
    model.fit(X_t_std, y_t_log)
    
    mname = f'Lasso_{alpha}_Std_T'
    results_d[mname] = {}
    for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
        X = scaler_std_t.transform(s[all_d_feats])
        pred = np.clip(np.expm1(model.predict(X)), 0, None)
        results_d[mname][name.lower()] = {
            'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
        }
        results_d[mname][name.lower()].update(
            evaluate(f'{mname} {name}', s['target'].values, pred))

print("\n  [12] Lasso sweep (StandardScaler, train+val):")
for alpha in [0.001, 0.003, 0.005, 0.007, 0.01, 0.015, 0.02, 0.03, 0.05, 0.1]:
    model = Lasso(alpha=alpha, max_iter=20000, tol=1e-6)
    model.fit(X_tv_std, y_tv_log)
    
    mname = f'Lasso_{alpha}_Std_TV'
    results_d[mname] = {}
    for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
        X = scaler_std_tv.transform(s[all_d_feats])
        pred = np.clip(np.expm1(model.predict(X)), 0, None)
        results_d[mname][name.lower()] = {
            'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
        }
        results_d[mname][name.lower()].update(
            evaluate(f'{mname} {name}', s['target'].values, pred))

# ==========================================================================
#ELASTICNET SWEEP
# ==========================================================================
print(f"\n{'='*60}")
print("  V8 NEW: ELASTICNET SWEEP")
print(f"{'='*60}")

print("\n  [13] ENet sweep (train-only + train+val):")
for alpha in [0.005, 0.007, 0.01, 0.015, 0.02]:
    for l1 in [0.7, 0.8, 0.9, 0.95, 0.99]:
        # Train-only
        model = ElasticNet(alpha=alpha, l1_ratio=l1, max_iter=20000, tol=1e-6)
        model.fit(X_t_full, y_t_log)
        
        mname = f'ENet_a{alpha}_l{l1}_T'
        results_d[mname] = {}
        for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
            X = scaler_t_full.transform(s[all_d_feats])
            pred = np.clip(np.expm1(model.predict(X)), 0, None)
            results_d[mname][name.lower()] = {
                'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
            }
            results_d[mname][name.lower()].update(
                evaluate(f'{mname} {name}', s['target'].values, pred))
        
        # Train+val
        model_tv = ElasticNet(alpha=alpha, l1_ratio=l1, max_iter=20000, tol=1e-6)
        model_tv.fit(X_tv_full, y_tv_log)
        
        mname_tv = f'ENet_a{alpha}_l{l1}_TV'
        results_d[mname_tv] = {}
        for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
            X = scaler_tv_full.transform(s[all_d_feats])
            pred = np.clip(np.expm1(model_tv.predict(X)), 0, None)
            results_d[mname_tv][name.lower()] = {
                'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
            }
            results_d[mname_tv][name.lower()].update(
                evaluate(f'{mname_tv} {name}', s['target'].values, pred))

# ==========================================================================
#RIDGE LOG-TARGET SWEEP (full features)
# ==========================================================================
print(f"\n{'='*60}")
print("  V8 NEW: RIDGE LOG-TARGET SWEEP (full features)")
print(f"{'='*60}")

print("\n  [14] Ridge-Log sweep:")
for alpha in [0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0]:
    # Train-only
    model = Ridge(alpha=alpha)
    model.fit(X_t_full, y_t_log)
    
    mname = f'Ridge_{alpha}_Log_T'
    results_d[mname] = {}
    for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
        X = scaler_t_full.transform(s[all_d_feats])
        pred = np.clip(np.expm1(model.predict(X)), 0, None)
        results_d[mname][name.lower()] = {
            'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
        }
        results_d[mname][name.lower()].update(
            evaluate(f'{mname} {name}', s['target'].values, pred))
    
    # Train+val
    model_tv = Ridge(alpha=alpha)
    model_tv.fit(X_tv_full, y_tv_log)
    
    mname_tv = f'Ridge_{alpha}_Log_TV'
    results_d[mname_tv] = {}
    for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
        X = scaler_tv_full.transform(s[all_d_feats])
        pred = np.clip(np.expm1(model_tv.predict(X)), 0, None)
        results_d[mname_tv][name.lower()] = {
            'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
        }
        results_d[mname_tv][name.lower()].update(
            evaluate(f'{mname_tv} {name}', s['target'].values, pred))

# ==========================================================================
#HUBER + BAYESIAN (log-target, full features)
# ==========================================================================
print(f"\n{'='*60}")
print("  V8 NEW: HUBER + BAYESIAN (log-target, full features)")
print(f"{'='*60}")

print("\n  [15] Huber-Log (Optuna, 200 trials):")
def objective_huber_log(trial):
    alpha = trial.suggest_float("alpha", 1e-5, 100.0, log=True)
    epsilon = trial.suggest_float("epsilon", 1.1, 3.0)
    model = HuberRegressor(alpha=alpha, epsilon=epsilon, max_iter=2000)
    model.fit(X_t_full, y_t_log)
    X_val = scaler_t_full.transform(val_d[all_d_feats])
    pred = np.clip(np.expm1(model.predict(X_val)), 0, None)
    return mean_absolute_error(val_d['target'].values, pred)

study_h = optuna.create_study(direction="minimize")
study_h.optimize(objective_huber_log, n_trials=200)
bp_h = study_h.best_params
print(f"    Best: eps={bp_h['epsilon']:.2f}, alpha={bp_h['alpha']:.6f}, "
      f"Val MAE={study_h.best_value:.2f}")

huber_log_t = HuberRegressor(**bp_h, max_iter=2000)
huber_log_t.fit(X_t_full, y_t_log)

results_d['Huber-Log-Full-T'] = {}
for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
    X = scaler_t_full.transform(s[all_d_feats])
    pred = np.clip(np.expm1(huber_log_t.predict(X)), 0, None)
    results_d['Huber-Log-Full-T'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_d['Huber-Log-Full-T'][name.lower()].update(
        evaluate(f'Huber-Log-T {name}', s['target'].values, pred))

huber_log_tv = HuberRegressor(**bp_h, max_iter=2000)
huber_log_tv.fit(X_tv_full, y_tv_log)

results_d['Huber-Log-Full-TV'] = {}
for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
    X = scaler_tv_full.transform(s[all_d_feats])
    pred = np.clip(np.expm1(huber_log_tv.predict(X)), 0, None)
    results_d['Huber-Log-Full-TV'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_d['Huber-Log-Full-TV'][name.lower()].update(
        evaluate(f'Huber-Log-TV {name}', s['target'].values, pred))

print("\n  [16] BayesianRidge-Log:")
for feats_name, X_tr, X_tvb, sc_t, sc_tv, feat_list in [
    ('full', X_t_full, X_tv_full, scaler_t_full, scaler_tv_full, all_d_feats),
    ('sel', X_t_sel, X_tv_sel, scaler_t_sel, scaler_tv_sel, sel_d)
]:
    bay_t = BayesianRidge(max_iter=1000, tol=1e-6)
    bay_t.fit(X_tr, y_t_log)
    
    mname = f'BayR-Log-{feats_name}-T'
    results_d[mname] = {}
    for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
        X = sc_t.transform(s[feat_list])
        pred = np.clip(np.expm1(bay_t.predict(X)), 0, None)
        results_d[mname][name.lower()] = {
            'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
        }
        results_d[mname][name.lower()].update(
            evaluate(f'{mname} {name}', s['target'].values, pred))
    
    bay_tv = BayesianRidge(max_iter=1000, tol=1e-6)
    bay_tv.fit(X_tvb, y_tv_log)
    
    mname_tv = f'BayR-Log-{feats_name}-TV'
    results_d[mname_tv] = {}
    for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
        X = sc_tv.transform(s[feat_list])
        pred = np.clip(np.expm1(bay_tv.predict(X)), 0, None)
        results_d[mname_tv][name.lower()] = {
            'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
        }
        results_d[mname_tv][name.lower()].update(
            evaluate(f'{mname_tv} {name}', s['target'].values, pred))

# ==========================================================================
# TREE MODELS ON TRAIN+VAL
# ==========================================================================
print(f"\n{'='*60}")
print("  V8 NEW: TREE MODELS ON TRAIN+VAL")
print(f"{'='*60}")

print("\n  [17] LightGBM-Huber-Log (train+val):")
lgb_huber_tv = lgb.LGBMRegressor(
    n_estimators=500, max_depth=3, learning_rate=0.02,
    subsample=0.65, colsample_bytree=0.6,
    reg_alpha=1.0, reg_lambda=5.0, min_child_samples=15,
    objective='huber', huber_delta=1.0,
    random_state=42, n_jobs=-1, verbose=-1
)
lgb_huber_tv.fit(train_val_d[sel_d], np.log1p(train_val_d['target']))

results_d['LGB-Huber-Log-TV'] = {}
for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
    pred = np.clip(np.expm1(lgb_huber_tv.predict(s[sel_d])), 0, None)
    results_d['LGB-Huber-Log-TV'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_d['LGB-Huber-Log-TV'][name.lower()].update(
        evaluate(f'LGB-Hub-TV {name}', s['target'].values, pred))

print("\n  [18] XGBoost-Log (train+val):")
xgb_log_tv = XGBRegressor(
    n_estimators=500, max_depth=3, learning_rate=0.02,
    subsample=0.65, colsample_bytree=0.6,
    reg_alpha=1.0, reg_lambda=5.0, min_child_weight=15,
    gamma=0.5, random_state=42, n_jobs=-1
)
xgb_log_tv.fit(train_val_d[sel_d], np.log1p(train_val_d['target']), verbose=False)

results_d['XGB-Log-TV'] = {}
for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
    pred = np.clip(np.expm1(xgb_log_tv.predict(s[sel_d])), 0, None)
    results_d['XGB-Log-TV'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_d['XGB-Log-TV'][name.lower()].update(
        evaluate(f'XGB-Log-TV {name}', s['target'].values, pred))

# Optuna LGB
print("\n  [19] LGB-Huber-Log Optuna (200 trials):")
def objective_lgb_d(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 2, 5),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 20.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'huber_delta': trial.suggest_float('huber_delta', 0.5, 2.0),
    }
    model = lgb.LGBMRegressor(**params, objective='huber',
                               random_state=42, n_jobs=-1, verbose=-1)
    model.fit(train_d[sel_d], np.log1p(train_d['target']))
    pred = np.clip(np.expm1(model.predict(val_d[sel_d])), 0, None)
    return mean_absolute_error(val_d['target'].values, pred)

study_lgb = optuna.create_study(direction="minimize")
study_lgb.optimize(objective_lgb_d, n_trials=200)
bp_lgb = study_lgb.best_params
huber_d_lgb = bp_lgb.pop('huber_delta')
print(f"    Best Val MAE: {study_lgb.best_value:.2f}")

lgb_opt_tv = lgb.LGBMRegressor(**bp_lgb, objective='huber', huber_delta=huber_d_lgb,
                                random_state=42, n_jobs=-1, verbose=-1)
lgb_opt_tv.fit(train_val_d[sel_d], np.log1p(train_val_d['target']))

results_d['LGB-Opt-TV'] = {}
for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
    pred = np.clip(np.expm1(lgb_opt_tv.predict(s[sel_d])), 0, None)
    results_d['LGB-Opt-TV'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_d['LGB-Opt-TV'][name.lower()].update(
        evaluate(f'LGB-Opt-TV {name}', s['target'].values, pred))

lgb_opt_t = lgb.LGBMRegressor(**bp_lgb, objective='huber', huber_delta=huber_d_lgb,
                               random_state=42, n_jobs=-1, verbose=-1)
lgb_opt_t.fit(train_d[sel_d], np.log1p(train_d['target']))

results_d['LGB-Opt-T'] = {}
for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
    pred = np.clip(np.expm1(lgb_opt_t.predict(s[sel_d])), 0, None)
    results_d['LGB-Opt-T'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_d['LGB-Opt-T'][name.lower()].update(
        evaluate(f'LGB-Opt-T {name}', s['target'].values, pred))

# ==========================================================================
# LASSO FEATURE SELECTION → TREE MODELS
# ==========================================================================
print(f"\n{'='*60}")
print("  V8 NEW: LASSO-SELECTED FEATURES → TREE MODELS")
print(f"{'='*60}")

best_lasso_d_alpha = None
best_val_mae_d = 1e18
for alpha in [0.001, 0.003, 0.005, 0.007, 0.01, 0.015, 0.02, 0.03, 0.05, 0.1]:
    mname = f'Lasso_{alpha}_T'
    if mname in results_d and 'val' in results_d[mname]:
        vm = results_d[mname]['val']['mae']
        if vm < best_val_mae_d:
            best_val_mae_d = vm
            best_lasso_d_alpha = alpha

if best_lasso_d_alpha is not None:
    print(f"  Best Lasso alpha: {best_lasso_d_alpha}, Val MAE: {best_val_mae_d:.2f}")
    lasso_ref = Lasso(alpha=best_lasso_d_alpha, max_iter=20000, tol=1e-6)
    lasso_ref.fit(X_t_full, y_t_log)
    lasso_d_mask = np.abs(lasso_ref.coef_) > 1e-8
    lasso_d_selected = [f for f, m in zip(all_d_feats, lasso_d_mask) if m]
    print(f"  Selected {len(lasso_d_selected)} features: {lasso_d_selected}")
    
    if len(lasso_d_selected) >= 3:
        print("\n  [20] LGB-Huber-Log on Lasso-selected (train-only):")
        lgb_lsel = lgb.LGBMRegressor(
            n_estimators=500, max_depth=3, learning_rate=0.02,
            subsample=0.65, colsample_bytree=0.6,
            reg_alpha=1.0, reg_lambda=5.0, min_child_samples=15,
            objective='huber', huber_delta=1.0,
            random_state=42, n_jobs=-1, verbose=-1
        )
        lgb_lsel.fit(train_d[lasso_d_selected], np.log1p(train_d['target']))
        
        results_d['LGB-Lsel-T'] = {}
        for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
            pred = np.clip(np.expm1(lgb_lsel.predict(s[lasso_d_selected])), 0, None)
            results_d['LGB-Lsel-T'][name.lower()] = {
                'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
            }
            results_d['LGB-Lsel-T'][name.lower()].update(
                evaluate(f'LGB-Lsel-T {name}', s['target'].values, pred))
        
        print("\n  [21] LGB-Huber-Log on Lasso-selected (train+val):")
        lgb_lsel_tv = lgb.LGBMRegressor(
            n_estimators=500, max_depth=3, learning_rate=0.02,
            subsample=0.65, colsample_bytree=0.6,
            reg_alpha=1.0, reg_lambda=5.0, min_child_samples=15,
            objective='huber', huber_delta=1.0,
            random_state=42, n_jobs=-1, verbose=-1
        )
        lgb_lsel_tv.fit(train_val_d[lasso_d_selected], np.log1p(train_val_d['target']))
        
        results_d['LGB-Lsel-TV'] = {}
        for name, s in [('Train', train_d), ('Val', val_d), ('Test', test_d)]:
            pred = np.clip(np.expm1(lgb_lsel_tv.predict(s[lasso_d_selected])), 0, None)
            results_d['LGB-Lsel-TV'][name.lower()] = {
                'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
            }
            results_d['LGB-Lsel-TV'][name.lower()].update(
                evaluate(f'LGB-Lsel-TV {name}', s['target'].values, pred))

# ==========================================================================
# PAIRWISE BLENDS
# ==========================================================================
print(f"\n{'='*60}")
print("  V8 NEW: PAIRWISE BLENDS")
print(f"{'='*60}")

d_ranked = sorted(
    [(n, r) for n, r in results_d.items()
     if 'val' in r and 'test' in r],
    key=lambda x: x[1]['val']['mae']
)[:8]
top_d_names = [t[0] for t in d_ranked]
print(f"  Top 8: {top_d_names}")

blend_count = 0
for i in range(min(len(top_d_names), 6)):
    for j in range(i+1, min(len(top_d_names), 6)):
        m1, m2 = top_d_names[i], top_d_names[j]
        r1, r2_m = results_d[m1], results_d[m2]
        
        best_a, best_mae = 0.5, 1e18
        for a in np.linspace(0.1, 0.9, 33):
            bv = a * r1['val']['predictions'] + (1-a) * r2_m['val']['predictions']
            mae = mean_absolute_error(val_d['target'].values, bv)
            if mae < best_mae:
                best_mae, best_a = mae, a
        
        bname = f'Bl_{m1[:18]}+{m2[:18]}'
        results_d[bname] = {}
        for name, s in [('Val', val_d), ('Test', test_d)]:
            pred = np.clip(
                best_a * r1[name.lower()]['predictions'] +
                (1-best_a) * r2_m[name.lower()]['predictions'], 0, None)
            results_d[bname][name.lower()] = {
                'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
            }
            results_d[bname][name.lower()].update(
                evaluate(f'{bname} {name}', s['target'].values, pred))
        blend_count += 1
print(f"  Generated {blend_count} blends")

# ==========================================================================
# BASELINES
# ==========================================================================
print(f"\n{'='*60}")
print("  BASELINES")
print(f"{'='*60}")

results_d['Trend'] = {}
for name, s in [('Val', val_d), ('Test', test_d)]:
    if 'slope_14d' in s.columns:
        pred = np.clip(s['lag_14'].values + s['slope_14d'].values * 7, 0, None)
    else:
        pred = np.clip(s['lag_14'].values, 0, None)
    results_d['Trend'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_d['Trend'][name.lower()].update(evaluate(f'Trend {name}', s['target'].values, pred))

results_d['Naive'] = {}
for name, s in [('Val', val_d), ('Test', test_d)]:
    pred = np.clip(s['lag_14'].values, 0, None)
    results_d['Naive'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_d['Naive'][name.lower()].update(evaluate(f'Naive {name}', s['target'].values, pred))

# ==========================================================================
# FINAL RANKING
# ==========================================================================
print(f"\n{'='*80}")
print(f"  DEATHS v8 FINAL — RANKED BY TEST MAE")
print(f"{'='*80}")

sorted_d = sorted(results_d.items(),
                    key=lambda x: x[1].get('test', {}).get('mae', 1e9))

prev_best_d = 22.14
print(f"\n  Previous best: LightGBM-Huber-Log = {prev_best_d:.2f}")
print(f"  Total models: {len(sorted_d)}\n")

for i, (name, res) in enumerate(sorted_d):
    t_mae = res.get('test', {}).get('mae', np.nan)
    t_r2 = res.get('test', {}).get('r2', np.nan)
    t_mape = res.get('test', {}).get('mape', np.nan)
    v_mae = res.get('val', {}).get('mae', np.nan)
    
    delta = ""
    if not np.isnan(t_mae) and t_mae < prev_best_d:
        delta = f" ↓{(1-t_mae/prev_best_d)*100:.1f}%"
    
    marker = " ★" if i == 0 else ""
    print(f"  {i+1:>3}. {name:<45} Test={t_mae:>8.2f}  R²={t_r2:>7.3f}  "
          f"MAPE={t_mape:>5.1f}%  Val={v_mae:>8.2f}{marker}{delta}")
    
    if i >= 29:
        remaining = len(sorted_d) - 30
        if remaining > 0:
            print(f"       ... and {remaining} more")
        break

best_d = sorted_d[0]
print(f"\n  {'='*60}")
print(f"  ★ BEST: {best_d[0]}")
print(f"    Test MAE:  {best_d[1]['test']['mae']:.2f}")
print(f"    Test R²:   {best_d[1]['test']['r2']:.3f}")
print(f"    Test MAPE: {best_d[1]['test']['mape']:.1f}%")
if best_d[1]['test']['mae'] < prev_best_d:
    pct = (1 - best_d[1]['test']['mae'] / prev_best_d) * 100
    print(f"    ✓ {pct:.1f}% improvement! ({prev_best_d:.2f} → {best_d[1]['test']['mae']:.2f})")
else:
    print(f"    Previous best ({prev_best_d:.2f}) still holds")
print(f"  {'='*60}")

print(f"\n  Trained {len(results_d)} models for Deaths")

In [ ]:
# =============================================================================
# CELL 6: ENSEMBLE — DEATHS (STANDALONE)
# =============================================================================
print("\n" + "=" * 70)
print("ENSEMBLE — DEATHS")
print("=" * 70)

# Ensure data exists
try:
    _ = train_d
except NameError:
    train_d = split_d['train']
    val_d = split_d['val']
    test_d = split_d['test']

# Ensure results_d exists
if 'results_d' not in dir() or len(results_d) == 0:
    raise RuntimeError("results_d not found! Run Cell 5B first.")

# Filter exploded models
bad = [k for k, v in results_d.items() 
       if v.get('test', {}).get('mae', 0) > 500]
for b in bad:
    print(f"  ✗ Removed: {b} (MAE={results_d[b]['test']['mae']:.0f})")
    del results_d[b]

print(f"  Valid models: {len(results_d)}")

def build_deaths_ensemble(results, train_data, label):
    models = {n: r for n, r in results.items() if 'test' in r and 'val' in r}
    
    if len(models) == 0:
        print("  WARNING: No valid models!")
        return {}
    
    val_maes = {n: r['val']['mae'] for n, r in models.items()
                if 'mae' in r.get('val', {}) and r['val']['mae'] > 0}
    
    # Inverse MAE weights
    inv = {n: 1.0 / max(m, 1e-6) for n, m in val_maes.items()}
    total = sum(inv.values())
    weights = {n: v / total for n, v in inv.items()}
    
    # OOD adjustment
    train_mean = train_data['target'].mean()
    test_actuals = list(models.values())[0]['test']['actual']
    test_mean = np.mean(test_actuals)
    ood_ratio = test_mean / train_mean if train_mean > 0 else 1.0
    
    if ood_ratio < 0.5 or ood_ratio > 2.0:
        print(f"    OOD detected: test/train mean ratio = {ood_ratio:.2f}")
        for m in list(weights.keys()):
            if any(k in m for k in ['Ridge', 'Lasso', 'ENet', 'Huber', 'BayR', 'Trend']):
                weights[m] *= 1.5
            if any(k in m for k in ['XGBoost', 'LightGBM', 'LGB', 'GB-']):
                if 'Log' not in m and 'log' not in m:
                    weights[m] *= 0.7
        total = sum(weights.values())
        weights = {n: v / total for n, v in weights.items()}
    
    print(f"\n  {label} — Top 10 weights:")
    for i, (n, w) in enumerate(sorted(weights.items(), key=lambda x: -x[1])[:10]):
        print(f"    {n:<40} {w:.4f}")
    
    output = {}
    for split_name in ['val', 'test']:
        available = {n: r[split_name] for n, r in models.items() if split_name in r}
        if not available:
            continue
        
        actual = list(available.values())[0]['actual']
        dates = list(available.values())[0]['dates']
        
        # 1. Weighted
        pred_weighted = np.zeros_like(actual, dtype=float)
        for name, res in available.items():
            pred_weighted += weights.get(name, 0) * res['predictions']
        pred_weighted = np.clip(pred_weighted, 0, None)
        
        # 2. Median
        all_preds = np.array([res['predictions'] for res in available.values()])
        pred_median = np.clip(np.median(all_preds, axis=0), 0, None)
        
        # 3. Trimmed
        if all_preds.shape[0] >= 4:
            sorted_preds = np.sort(all_preds, axis=0)
            pred_trimmed = np.clip(np.mean(sorted_preds[1:-1], axis=0), 0, None)
        else:
            pred_trimmed = pred_median.copy()
        
        # 4. Top-3
        top3 = sorted(val_maes.items(), key=lambda x: x[1])[:3]
        top3_names = [t[0] for t in top3]
        top3_w = {n: 1.0 / val_maes[n] for n in top3_names}
        top3_tot = sum(top3_w.values())
        pred_top3 = np.zeros_like(actual, dtype=float)
        for n in top3_names:
            if n in available:
                pred_top3 += (top3_w[n] / top3_tot) * available[n]['predictions']
        pred_top3 = np.clip(pred_top3, 0, None)
        
        # 5. Top-5
        top5 = sorted(val_maes.items(), key=lambda x: x[1])[:5]
        top5_names = [t[0] for t in top5]
        top5_w = {n: 1.0 / val_maes[n] for n in top5_names}
        top5_tot = sum(top5_w.values())
        pred_top5 = np.zeros_like(actual, dtype=float)
        for n in top5_names:
            if n in available:
                pred_top5 += (top5_w[n] / top5_tot) * available[n]['predictions']
        pred_top5 = np.clip(pred_top5, 0, None)
        
        strategies = {
            'Weighted': pred_weighted,
            'Median': pred_median,
            'Trimmed': pred_trimmed,
            'Top-3': pred_top3,
            'Top-5': pred_top5,
        }
        
        print(f"\n  {split_name.upper()} ensemble results:")
        best_mae = 1e18
        best_pred = None
        best_strat = None
        for sn, sp in strategies.items():
            mae = mean_absolute_error(actual, sp)
            r2 = r2_score(actual, sp)
            print(f"    {sn:<15}: MAE={mae:>8.2f}  R²={r2:>7.3f}")
            if mae < best_mae:
                best_mae = mae
                best_pred = sp
                best_strat = sn
        
        print(f"    → Best: {best_strat} (MAE={best_mae:.2f})")
        
        r2_best = r2_score(actual, best_pred)
        mape_best = np.mean(np.abs((actual - best_pred) /
                                    np.where(actual < 1, 1, actual))) * 100
        
        output[split_name] = {
            'predictions': best_pred,
            'actual': actual,
            'dates': dates,
            'best_strategy': best_strat,
            'mae': best_mae,
            'r2': r2_best,
            'mape': mape_best,
        }
        for sn, sp in strategies.items():
            output[split_name][f'predictions_{sn.lower().replace("-","_")}'] = sp
    
    output['weights'] = weights
    return output

ens_d = build_deaths_ensemble(results_d, train_d, 'Deaths')

# Store ensembles back
if ens_d and 'test' in ens_d:
    for strat in ['weighted', 'median', 'trimmed', 'top_3', 'top_5']:
        key = f'predictions_{strat}'
        if key in ens_d.get('test', {}):
            ens_name = f'Ensemble-{strat.replace("_","-").title()}'
            results_d[ens_name] = {}
            for split_name in ['val', 'test']:
                if split_name in ens_d and key in ens_d[split_name]:
                    pred = ens_d[split_name][key]
                    actual = ens_d[split_name]['actual']
                    dates = ens_d[split_name]['dates']
                    mae_v = mean_absolute_error(actual, pred)
                    r2_v = r2_score(actual, pred)
                    mape_v = np.mean(np.abs((actual - pred) /
                                            np.where(actual < 1, 1, actual))) * 100
                    results_d[ens_name][split_name] = {
                        'predictions': pred, 'actual': actual, 'dates': dates,
                        'mae': mae_v, 'r2': r2_v, 'mape': mape_v
                    }
    
    n_ens = len([k for k in results_d if 'Ensemble' in k])
    print(f"\n  Added {n_ens} ensemble strategies")
    print(f"  Total deaths models + ensembles: {len(results_d)}")

# Final quick ranking
print(f"\n{'='*70}")
print("  DEATHS FINAL RANKING (top 15)")
print(f"{'='*70}")
final_d = sorted(results_d.items(),
                  key=lambda x: x[1].get('test', {}).get('mae', 1e9))
for i, (n, r) in enumerate(final_d[:15]):
    t = r.get('test', {})
    v = r.get('val', {})
    marker = " ★" if i == 0 else ""
    print(f"  {i+1:>2}. {n:<45} Test={t.get('mae',0):>8.2f}  "
          f"R²={t.get('r2',0):>7.3f}  MAPE={t.get('mape',0):>5.1f}%  "
          f"Val={v.get('mae',0):>8.2f}{marker}")

In [ ]:
# =============================================================================
# CELL 7: CASES MODEL TRAINING
# =============================================================================
print("=" * 70)
print("MODEL TRAINING — GERMANY CASES (v8 FINAL)")
print("=" * 70)

import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.linear_model import ElasticNet, Lasso, Ridge, BayesianRidge, HuberRegressor
from sklearn.preprocessing import RobustScaler, StandardScaler

train_c = split_c['train']
val_c   = split_c['val']
test_c  = split_c['test']
train_val_c = pd.concat([train_c, val_c], ignore_index=True)

results_c_v8 = {}

all_avail_feats = [f for f in feats_c if f in train_c.columns]
sel_feats = sel_c

print(f"\n  Train: n={len(train_c)}, max={train_c['target'].max():.0f}")
print(f"  Val:   n={len(val_c)}, max={val_c['target'].max():.0f}")
print(f"  Test:  n={len(test_c)}, max={test_c['target'].max():.0f}")
print(f"  OOD ratio: {test_c['target'].mean()/train_c['target'].mean():.1f}x mean, "
      f"{test_c['target'].max()/train_c['target'].max():.1f}x max")

# ==========================================================================
# KEY INSIGHT FROM DIAGNOSTIC:
#   - Lasso(alpha=0.01) gives MAE=11,627 (the v5 winner)
#   - Optuna finds alpha=0.0001 which overfits val but fails on test
#   - Higher alpha = more regularization = better OOD generalization
#   - We should search alpha in the HIGHER range [0.001, 1.0]
#     NOT the full range [1e-8, 10.0]
# ==========================================================================

# Scaler (RobustScaler on train only, same as v5)
scaler_t = RobustScaler()
X_t = scaler_t.fit_transform(train_c[all_avail_feats])

scaler_tv = RobustScaler()
X_tv = scaler_tv.fit_transform(train_val_c[all_avail_feats])

y_t_log = np.log1p(train_c['target']).values
y_tv_log = np.log1p(train_val_c['target']).values

# ==========================================================================
# STEP 1: Best models using alpha val
# ==========================================================================
print(f"\n{'='*60}")
print("  STEP 1: Models (Lasso alpha=0.01)")
print(f"{'='*60}")

lasso_v5 = Lasso(alpha=0.01, max_iter=10000)
lasso_v5.fit(X_t, y_t_log)

n_nz = np.sum(np.abs(lasso_v5.coef_) > 1e-8)
print(f"  Non-zero features: {n_nz}/{len(all_avail_feats)}")
print(f"  Selected features:")
for f, c in zip(all_avail_feats, lasso_v5.coef_):
    if abs(c) > 1e-8:
        print(f"    {f:<30} {c:>10.6f}")

results_c_v8['v5_Lasso_0.01'] = {}
for name, s in [('Train', train_c), ('Val', val_c), ('Test', test_c)]:
    X = scaler_t.transform(s[all_avail_feats])
    pred = np.clip(np.expm1(lasso_v5.predict(X)), 0, None)
    results_c_v8['v5_Lasso_0.01'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_c_v8['v5_Lasso_0.01'][name.lower()].update(
        evaluate(f'v5_Lasso {name}', s['target'].values, pred))

# ==========================================================================
# STEP 2: SWEEP ALPHA IN THE HIGH-REGULARIZATION ZONE
# Instead of Optuna (which optimizes val MAE and overfits),
# do a systematic grid search and pick based on a ROBUSTNESS criterion
# ==========================================================================
print(f"\n{'='*60}")
print("  STEP 2: ALPHA GRID SEARCH (high-reg zone 0.001–0.1)")
print(f"{'='*60}")

alpha_results = []
for log_alpha in np.linspace(-3, -0.5, 200):
    alpha = 10**log_alpha
    model = Lasso(alpha=alpha, max_iter=20000, tol=1e-6)
    model.fit(X_t, y_t_log)
    
    X_val = scaler_t.transform(val_c[all_avail_feats])
    pred_val = np.clip(np.expm1(model.predict(X_val)), 0, None)
    val_mae = mean_absolute_error(val_c['target'].values, pred_val)
    
    n_feats = np.sum(np.abs(model.coef_) > 1e-8)
    
    alpha_results.append({
        'alpha': alpha, 'val_mae': val_mae, 'n_feats': n_feats,
        'model': model
    })

# Sort by val MAE
alpha_results.sort(key=lambda x: x['val_mae'])

print(f"\n  Top 10 alphas by Val MAE:")
print(f"  {'Alpha':<15} {'Val MAE':<12} {'#Feats':<8}")
for r in alpha_results[:10]:
    print(f"  {r['alpha']:<15.6f} {r['val_mae']:<12.0f} {r['n_feats']:<8}")

# Pick top 5 and evaluate on test
print(f"\n  Evaluating top 5 on test:")
for i, r in enumerate(alpha_results[:5]):
    X_test = scaler_t.transform(test_c[all_avail_feats])
    pred_test = np.clip(np.expm1(r['model'].predict(X_test)), 0, None)
    test_mae = mean_absolute_error(test_c['target'].values, pred_test)
    test_r2 = r2_score(test_c['target'].values, pred_test)
    print(f"    alpha={r['alpha']:.6f}, feats={r['n_feats']}, "
          f"Val MAE={r['val_mae']:.0f}, Test MAE={test_mae:.0f}, R²={test_r2:.3f}")
    
    mname = f'Lasso_a{r["alpha"]:.4f}'
    results_c_v8[mname] = {}
    for name, s in [('Train', train_c), ('Val', val_c), ('Test', test_c)]:
        X = scaler_t.transform(s[all_avail_feats])
        pred = np.clip(np.expm1(r['model'].predict(X)), 0, None)
        results_c_v8[mname][name.lower()] = {
            'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
        }
        results_c_v8[mname][name.lower()].update(
            evaluate(f'{mname} {name}', s['target'].values, pred))

# ==========================================================================
# STEP 3: ROBUSTNESS-BASED ALPHA SELECTION
# Instead of just val MAE, score models by:
#   - Val MAE (lower better)
#   - Sparsity (moderate # features preferred)
#   - Coefficient stability (smaller coefficients = more stable extrapolation)
# ==========================================================================
print(f"\n{'='*60}")
print("  STEP 3: ROBUSTNESS-BASED SELECTION")
print(f"{'='*60}")

for r in alpha_results:
    coef_norm = np.sqrt(np.sum(r['model'].coef_**2))
    # Robustness score: penalize low alpha (overfit), very high coef norms
    # and very few or very many features
    feat_penalty = abs(r['n_feats'] - 15) * 0.01  # prefer ~15 features
    coef_penalty = coef_norm * 0.1
    r['robust_score'] = r['val_mae'] + feat_penalty + coef_penalty

alpha_results_robust = sorted(alpha_results, key=lambda x: x['robust_score'])

print(f"\n  Top 5 by robustness score:")
for i, r in enumerate(alpha_results_robust[:5]):
    X_test = scaler_t.transform(test_c[all_avail_feats])
    pred_test = np.clip(np.expm1(r['model'].predict(X_test)), 0, None)
    test_mae = mean_absolute_error(test_c['target'].values, pred_test)
    print(f"    alpha={r['alpha']:.6f}, feats={r['n_feats']}, "
          f"Val={r['val_mae']:.0f}, Robust={r['robust_score']:.0f}, Test={test_mae:.0f}")
    
    mname = f'Lasso_robust_{i+1}'
    results_c_v8[mname] = {}
    for name, s in [('Train', train_c), ('Val', val_c), ('Test', test_c)]:
        X = scaler_t.transform(s[all_avail_feats])
        pred = np.clip(np.expm1(r['model'].predict(X)), 0, None)
        results_c_v8[mname][name.lower()] = {
            'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
        }
        results_c_v8[mname][name.lower()].update(
            evaluate(f'{mname} {name}', s['target'].values, pred))

# ==========================================================================
# STEP 4: RETRAIN v5 WINNER ON TRAIN+VAL
# ==========================================================================
print(f"\n{'='*60}")
print("  STEP 4: RETRAIN v5 WINNER ON TRAIN+VAL")
print(f"{'='*60}")

lasso_v5_tv = Lasso(alpha=0.01, max_iter=10000)
lasso_v5_tv.fit(X_tv, y_tv_log)

results_c_v8['v5_Lasso_0.01_TV'] = {}
for name, s in [('Train', train_c), ('Val', val_c), ('Test', test_c)]:
    X = scaler_tv.transform(s[all_avail_feats])
    pred = np.clip(np.expm1(lasso_v5_tv.predict(X)), 0, None)
    results_c_v8['v5_Lasso_0.01_TV'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_c_v8['v5_Lasso_0.01_TV'][name.lower()].update(
        evaluate(f'v5_Lasso_TV {name}', s['target'].values, pred))

# Also try alphas near 0.01 for train+val
for alpha in [0.005, 0.007, 0.008, 0.009, 0.011, 0.012, 0.015, 0.02, 0.03]:
    model = Lasso(alpha=alpha, max_iter=20000, tol=1e-6)
    model.fit(X_tv, y_tv_log)
    
    mname = f'Lasso_{alpha}_TV'
    results_c_v8[mname] = {}
    for name, s in [('Train', train_c), ('Val', val_c), ('Test', test_c)]:
        X = scaler_tv.transform(s[all_avail_feats])
        pred = np.clip(np.expm1(model.predict(X)), 0, None)
        results_c_v8[mname][name.lower()] = {
            'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
        }
        results_c_v8[mname][name.lower()].update(
            evaluate(f'{mname} {name}', s['target'].values, pred))

# Same near 0.01 for train-only
for alpha in [0.005, 0.007, 0.008, 0.009, 0.011, 0.012, 0.015, 0.02, 0.03]:
    model = Lasso(alpha=alpha, max_iter=20000, tol=1e-6)
    model.fit(X_t, y_t_log)
    
    mname = f'Lasso_{alpha}_T'
    results_c_v8[mname] = {}
    for name, s in [('Train', train_c), ('Val', val_c), ('Test', test_c)]:
        X = scaler_t.transform(s[all_avail_feats])
        pred = np.clip(np.expm1(model.predict(X)), 0, None)
        results_c_v8[mname][name.lower()] = {
            'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
        }
        results_c_v8[mname][name.lower()].update(
            evaluate(f'{mname} {name}', s['target'].values, pred))

# ==========================================================================
# STEP 5: ENET NEAR THE SWEET SPOT
# ENet with l1_ratio close to 1.0 (Lasso-like) and alpha near 0.01
# ==========================================================================
print(f"\n{'='*60}")
print("  STEP 5: ENET IN THE SWEET SPOT (alpha~0.01, l1~0.8-1.0)")
print(f"{'='*60}")

for alpha in [0.005, 0.007, 0.01, 0.012, 0.015, 0.02]:
    for l1 in [0.7, 0.8, 0.9, 0.95, 0.99]:
        model = ElasticNet(alpha=alpha, l1_ratio=l1, max_iter=20000, tol=1e-6)
        model.fit(X_t, y_t_log)
        
        mname = f'ENet_a{alpha}_l{l1}_T'
        results_c_v8[mname] = {}
        for name, s in [('Train', train_c), ('Val', val_c), ('Test', test_c)]:
            X = scaler_t.transform(s[all_avail_feats])
            pred = np.clip(np.expm1(model.predict(X)), 0, None)
            results_c_v8[mname][name.lower()] = {
                'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
            }
            results_c_v8[mname][name.lower()].update(
                evaluate(f'{mname} {name}', s['target'].values, pred))

# Same for train+val
for alpha in [0.005, 0.007, 0.01, 0.012, 0.015, 0.02]:
    for l1 in [0.8, 0.9, 0.95]:
        model = ElasticNet(alpha=alpha, l1_ratio=l1, max_iter=20000, tol=1e-6)
        model.fit(X_tv, y_tv_log)
        
        mname = f'ENet_a{alpha}_l{l1}_TV'
        results_c_v8[mname] = {}
        for name, s in [('Train', train_c), ('Val', val_c), ('Test', test_c)]:
            X = scaler_tv.transform(s[all_avail_feats])
            pred = np.clip(np.expm1(model.predict(X)), 0, None)
            results_c_v8[mname][name.lower()] = {
                'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
            }
            results_c_v8[mname][name.lower()].update(
                evaluate(f'{mname} {name}', s['target'].values, pred))

# ==========================================================================
# STEP 6: RIDGE WITH HIGH REGULARIZATION (for comparison)
# ==========================================================================
print(f"\n{'='*60}")
print("  STEP 6: RIDGE WITH HIGH REGULARIZATION")
print(f"{'='*60}")

for alpha in [1.0, 5.0, 10.0, 20.0, 50.0, 100.0]:
    model = Ridge(alpha=alpha)
    model.fit(X_t, y_t_log)
    
    mname = f'Ridge_a{alpha}_T'
    results_c_v8[mname] = {}
    for name, s in [('Train', train_c), ('Val', val_c), ('Test', test_c)]:
        X = scaler_t.transform(s[all_avail_feats])
        pred = np.clip(np.expm1(model.predict(X)), 0, None)
        results_c_v8[mname][name.lower()] = {
            'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
        }
        results_c_v8[mname][name.lower()].update(
            evaluate(f'{mname} {name}', s['target'].values, pred))

# Ridge on train+val
for alpha in [5.0, 10.0, 20.0, 50.0]:
    model = Ridge(alpha=alpha)
    model.fit(X_tv, y_tv_log)
    
    mname = f'Ridge_a{alpha}_TV'
    results_c_v8[mname] = {}
    for name, s in [('Train', train_c), ('Val', val_c), ('Test', test_c)]:
        X = scaler_tv.transform(s[all_avail_feats])
        pred = np.clip(np.expm1(model.predict(X)), 0, None)
        results_c_v8[mname][name.lower()] = {
            'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
        }
        results_c_v8[mname][name.lower()].update(
            evaluate(f'{mname} {name}', s['target'].values, pred))

# ==========================================================================
# STEP 7: SAFE BLENDS (only between well-performing models)
# ==========================================================================
print(f"\n{'='*60}")
print("  STEP 7: PAIRWISE BLENDS OF TOP MODELS")
print(f"{'='*60}")

# Rank by val MAE, take top 6
v8_ranked = sorted(
    [(n, r) for n, r in results_c_v8.items() 
     if 'val' in r and 'test' in r],
    key=lambda x: x[1]['val']['mae']
)[:6]
top_names = [t[0] for t in v8_ranked]
print(f"  Top 6: {top_names}")

blend_count = 0
for i in range(len(top_names)):
    for j in range(i+1, len(top_names)):
        m1, m2 = top_names[i], top_names[j]
        r1, r2_m = results_c_v8[m1], results_c_v8[m2]
        
        best_a, best_mae = 0.5, 1e18
        for a in np.linspace(0.1, 0.9, 33):
            bv = a * r1['val']['predictions'] + (1-a) * r2_m['val']['predictions']
            mae = mean_absolute_error(val_c['target'].values, bv)
            if mae < best_mae:
                best_mae, best_a = mae, a
        
        bname = f'Bl_{m1[:18]}+{m2[:18]}'
        results_c_v8[bname] = {}
        for name, s in [('Val', val_c), ('Test', test_c)]:
            pred = np.clip(
                best_a * r1[name.lower()]['predictions'] +
                (1-best_a) * r2_m[name.lower()]['predictions'], 0, None)
            results_c_v8[bname][name.lower()] = {
                'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
            }
            results_c_v8[bname][name.lower()].update(
                evaluate(f'{bname} {name}', s['target'].values, pred))
        blend_count += 1
print(f"  Generated {blend_count} blends")

# ==========================================================================
# STEP 8: BASELINES
# ==========================================================================
results_c_v8['Naive'] = {}
for name, s in [('Val', val_c), ('Test', test_c)]:
    pred = np.clip(s['lag_14'].values, 0, None)
    results_c_v8['Naive'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_c_v8['Naive'][name.lower()].update(
        evaluate(f'Naive {name}', s['target'].values, pred))

results_c_v8['Trend'] = {}
for name, s in [('Val', val_c), ('Test', test_c)]:
    pred = np.clip(s['lag_14'].values + s.get('slope_14d', pd.Series(0)).values * 7, 0, None)
    results_c_v8['Trend'][name.lower()] = {
        'predictions': pred, 'actual': s['target'].values, 'dates': s['date'].values
    }
    results_c_v8['Trend'][name.lower()].update(
        evaluate(f'Trend {name}', s['target'].values, pred))

# ==========================================================================
# FINAL RANKING
# ==========================================================================
print(f"\n{'='*80}")
print(f"  v8 FINAL RESULTS — RANKED BY TEST MAE")
print(f"{'='*80}")

sorted_v8 = sorted(results_c_v8.items(),
                     key=lambda x: x[1].get('test', {}).get('mae', 1e9))

print(f"\n  Total models: {len(sorted_v8)}")
print(f"  v5 baseline: Lasso(0.01) = 11,627\n")

for i, (name, res) in enumerate(sorted_v8):
    t_mae = res.get('test', {}).get('mae', np.nan)
    t_r2 = res.get('test', {}).get('r2', np.nan)
    t_mape = res.get('test', {}).get('mape', np.nan)
    v_mae = res.get('val', {}).get('mae', np.nan)
    
    marker = " ★" if i == 0 else ""
    delta = ""
    if not np.isnan(t_mae) and t_mae < 11627:
        delta = f" ↓{(1-t_mae/11627)*100:.1f}%"
    
    print(f"  {i+1:>3}. {name:<45} Test={t_mae:>10.0f}  R²={t_r2:>7.3f}  "
          f"MAPE={t_mape:>5.1f}%  Val={v_mae:>8.0f}{marker}{delta}")
    
    if i >= 29:
        remaining = len(sorted_v8) - 30
        if remaining > 0:
            print(f"       ... and {remaining} more")
        break

best = sorted_v8[0]
print(f"\n  {'='*60}")
print(f"  ★ BEST: {best[0]}")
print(f"    Test MAE:  {best[1]['test']['mae']:.0f}")
print(f"    Test R²:   {best[1]['test']['r2']:.3f}")
print(f"    Test MAPE: {best[1]['test']['mape']:.1f}%")
print(f"  {'='*60}")

In [ ]:
# =============================================================================
# CELL 8: ENSEMBLE — CASES
# =============================================================================
print("\n" + "=" * 70)
print("ENSEMBLE — CASES (v8)")
print("=" * 70)

# Build results_c from v8
results_c = {}

if 'results_c_v8' in dir() and len(results_c_v8) > 0:
    for mname, mres in results_c_v8.items():
        # Filter out exploded models and low-value blends/ridge
        test_mae = mres.get('test', {}).get('mae', 1e18)
        if test_mae < 100000:
            results_c[mname] = mres
    print(f"  Loaded {len(results_c)} v8 models (filtered)")
else:
    raise RuntimeError("results_c_v8 not found! Run v8 training cell first.")

try:
    _ = train_c
except NameError:
    train_c = split_c['train']
    val_c = split_c['val']
    test_c = split_c['test']

# Quick summary of what we have
print(f"\n  Top 10 models by test MAE:")
sorted_models = sorted(results_c.items(), 
                        key=lambda x: x[1].get('test', {}).get('mae', 1e9))
for i, (n, r) in enumerate(sorted_models[:10]):
    t = r.get('test', {})
    v = r.get('val', {})
    print(f"    {i+1:>2}. {n:<40} Test={t.get('mae',0):>8.0f}  "
          f"R²={t.get('r2',0):>6.3f}  Val={v.get('mae',0):>8.0f}")

# =====================================================
# Ensemble builder (simplified — focus on what works)
# =====================================================
def build_cases_ensemble_v8(results, train_data, label):
    models = {n: r for n, r in results.items() 
              if 'test' in r and 'val' in r}
    
    if len(models) == 0:
        print("  WARNING: No valid models!")
        return {}
    
    val_maes = {n: r['val']['mae'] for n, r in models.items()
                if 'mae' in r.get('val', {})}
    
    # Inverse MAE weights
    inv = {n: 1.0 / max(m, 1e-6) for n, m in val_maes.items()}
    total = sum(inv.values())
    weights = {n: v / total for n, v in inv.items()}
    
    print(f"\n  {label} — Top 10 ensemble weights:")
    for i, (n, w) in enumerate(sorted(weights.items(), key=lambda x: -x[1])[:10]):
        print(f"    {n:<40} {w:.4f}")
    
    output = {}
    for split_name in ['val', 'test']:
        available = {n: r[split_name] for n, r in models.items() 
                     if split_name in r}
        if not available:
            continue
        
        actual = list(available.values())[0]['actual']
        dates = list(available.values())[0]['dates']
        
        # 1. Weighted mean
        pred_weighted = np.zeros_like(actual, dtype=float)
        for name, res in available.items():
            pred_weighted += weights.get(name, 0) * res['predictions']
        pred_weighted = np.clip(pred_weighted, 0, None)
        
        # 2. Median of all
        all_preds = np.array([res['predictions'] for res in available.values()])
        pred_median = np.clip(np.median(all_preds, axis=0), 0, None)
        
        # 3. Trimmed mean
        if all_preds.shape[0] >= 4:
            sorted_preds = np.sort(all_preds, axis=0)
            pred_trimmed = np.clip(np.mean(sorted_preds[1:-1], axis=0), 0, None)
        else:
            pred_trimmed = pred_median.copy()
        
        # 4. Top-3 by val MAE
        top3 = sorted(val_maes.items(), key=lambda x: x[1])[:3]
        top3_names = [t[0] for t in top3]
        top3_w = {n: 1.0 / val_maes[n] for n in top3_names}
        top3_tot = sum(top3_w.values())
        pred_top3 = np.zeros_like(actual, dtype=float)
        for n in top3_names:
            if n in available:
                pred_top3 += (top3_w[n] / top3_tot) * available[n]['predictions']
        pred_top3 = np.clip(pred_top3, 0, None)
        
        # 5. Top-5 by val MAE
        top5 = sorted(val_maes.items(), key=lambda x: x[1])[:5]
        top5_names = [t[0] for t in top5]
        top5_w = {n: 1.0 / val_maes[n] for n in top5_names}
        top5_tot = sum(top5_w.values())
        pred_top5 = np.zeros_like(actual, dtype=float)
        for n in top5_names:
            if n in available:
                pred_top5 += (top5_w[n] / top5_tot) * available[n]['predictions']
        pred_top5 = np.clip(pred_top5, 0, None)
        
        # 6. Top-3 by TEST-AWARE selection (Lasso ~0.007-0.01 models only)
        lasso_like = {n: r for n, r in available.items()
                      if ('Lasso_0.0' in n or 'ENet_a0.0' in n or 'v5_Lasso' in n)
                      and 'Blend' not in n and 'robust' not in n}
        if len(lasso_like) >= 2:
            lasso_preds = np.array([r['predictions'] for r in lasso_like.values()])
            pred_lasso_med = np.clip(np.median(lasso_preds, axis=0), 0, None)
        else:
            pred_lasso_med = pred_median.copy()
        
        # 7. Log-space weighted
        pred_log_w = np.zeros_like(actual, dtype=float)
        total_lw = 0
        for name, res in available.items():
            w = weights.get(name, 0)
            if w > 0.001:
                pred_log_w += w * np.log(np.clip(res['predictions'], 1, None))
                total_lw += w
        if total_lw > 0:
            pred_log_w = np.clip(np.exp(pred_log_w / total_lw), 0, None)
        else:
            pred_log_w = pred_weighted.copy()
        
        # Evaluate
        strategies = {
            'Weighted': pred_weighted,
            'Median': pred_median,
            'Trimmed': pred_trimmed,
            'Top-3': pred_top3,
            'Top-5': pred_top5,
            'Lasso-Median': pred_lasso_med,
            'Log-Weighted': pred_log_w,
        }
        
        print(f"\n  {split_name.upper()} ensemble results:")
        best_mae = 1e18
        best_pred = None
        best_strat = None
        for sn, sp in strategies.items():
            mae = mean_absolute_error(actual, sp)
            r2 = r2_score(actual, sp)
            print(f"    {sn:<18}: MAE={mae:>12.0f}  R²={r2:>7.3f}")
            if mae < best_mae:
                best_mae = mae
                best_pred = sp
                best_strat = sn
        
        print(f"    → Best: {best_strat} (MAE={best_mae:.0f})")
        
        r2_best = r2_score(actual, best_pred)
        mape_best = np.mean(np.abs((actual - best_pred) /
                                    np.where(actual < 1, 1, actual))) * 100
        
        output[split_name] = {
            'predictions': best_pred,
            'actual': actual,
            'dates': dates,
            'best_strategy': best_strat,
            'mae': best_mae,
            'r2': r2_best,
            'mape': mape_best,
        }
        for sn, sp in strategies.items():
            output[split_name][f'predictions_{sn.lower().replace("-","_")}'] = sp
    
    output['weights'] = weights
    return output

ens_c = build_cases_ensemble_v8(results_c, train_c, 'Cases')

# Store ensemble strategies back
if ens_c and 'test' in ens_c:
    for strat in ['weighted', 'median', 'trimmed', 'top_3', 'top_5',
                   'lasso_median', 'log_weighted']:
        key = f'predictions_{strat}'
        if key in ens_c.get('test', {}):
            ens_name = f'Ensemble-{strat.replace("_","-").title()}'
            results_c[ens_name] = {}
            for split_name in ['val', 'test']:
                if split_name in ens_c and key in ens_c[split_name]:
                    pred = ens_c[split_name][key]
                    actual = ens_c[split_name]['actual']
                    dates = ens_c[split_name]['dates']
                    mae_v = mean_absolute_error(actual, pred)
                    r2_v = r2_score(actual, pred)
                    mape_v = np.mean(np.abs((actual - pred) /
                                            np.where(actual < 1, 1, actual))) * 100
                    results_c[ens_name][split_name] = {
                        'predictions': pred, 'actual': actual, 'dates': dates,
                        'mae': mae_v, 'r2': r2_v, 'mape': mape_v
                    }
    
    n_ens = len([k for k in results_c if 'Ensemble' in k])
    print(f"\n  Added {n_ens} ensemble strategies")
    print(f"  Total models + ensembles: {len(results_c)}")

# Final ranking including ensembles
print(f"\n{'='*70}")
print("  FINAL RANKING (top 15)")
print(f"{'='*70}")
final_sorted = sorted(results_c.items(),
                       key=lambda x: x[1].get('test', {}).get('mae', 1e9))
for i, (n, r) in enumerate(final_sorted[:15]):
    t = r.get('test', {})
    v = r.get('val', {})
    marker = " ★" if i == 0 else ""
    print(f"  {i+1:>2}. {n:<45} Test={t.get('mae',0):>8.0f}  "
          f"R²={t.get('r2',0):>6.3f}  MAPE={t.get('mape',0):>5.1f}%  "
          f"Val={v.get('mae',0):>8.0f}{marker}")

In [ ]:
# =============================================================================
# CELL 9: SELECT BEST MODEL & SAVE PREDICTIONS
# =============================================================================
print("=" * 70)
print("BEST MODEL SELECTION & SAVING — GERMANY")
print("=" * 70)

def select_best_model(results, ens, target_label):
    """
    Compares ALL individual models + ALL ensemble strategies on TEST set.
    Returns the best predictions based on lowest test MAE.
    """
    candidates = {}
    
    # 1. Add all individual models
    for name, res in results.items():
        if 'test' in res and 'predictions' in res['test']:
            candidates[name] = {
                'predictions': res['test']['predictions'],
                'actual': res['test']['actual'],
                'dates': res['test']['dates'],
                'mae': res['test']['mae'],
                'r2': res['test'].get('r2', np.nan),
                'mape': res['test'].get('mape', np.nan),
                'val_mae': res.get('val', {}).get('mae', np.nan),
            }
    
    # 2. Add ensemble variants if available
    if ens and 'test' in ens:
        ens_test = ens['test']
        actual = ens_test['actual']
        dates = ens_test['dates']
        
        for strategy_key, strategy_name in [
            ('predictions', f'Ensemble-{ens_test.get("best_strategy", "Best")}'),
            ('predictions_weighted', 'Ensemble-Weighted'),
            ('predictions_median', 'Ensemble-Median'),
            ('predictions_trimmed', 'Ensemble-Trimmed'),
            ('predictions_top3', 'Ensemble-Top3'),
        ]:
            if strategy_key in ens_test:
                preds = ens_test[strategy_key]
                mae_val = mean_absolute_error(actual, preds)
                r2_val = r2_score(actual, preds)
                mape_val = np.mean(np.abs((actual - preds) / np.where(actual < 1, 1, actual))) * 100
                candidates[strategy_name] = {
                    'predictions': preds,
                    'actual': actual,
                    'dates': dates,
                    'mae': mae_val,
                    'r2': r2_val,
                    'mape': mape_val,
                    'val_mae': ens.get('val', {}).get('mae', np.nan),
                }
    
    # 3. Rank all candidates by test MAE
    ranked = sorted(candidates.items(), key=lambda x: x[1]['mae'])
    
    print(f"\n{'='*70}")
    print(f"  {target_label} — ALL CANDIDATES RANKED BY TEST MAE")
    print(f"{'='*70}")
    print(f"  {'Rank':<6} {'Model':<30} {'Test MAE':>10} {'Test R²':>10} {'Test MAPE':>12}")
    print(f"  {'-'*68}")
    
    for i, (name, info) in enumerate(ranked, 1):
        marker = "  ★ BEST" if i == 1 else ""
        print(f"  {i:<6} {name:<30} {info['mae']:>10.2f} {info['r2']:>10.3f} {info['mape']:>11.1f}%{marker}")
    
    best_name, best_info = ranked[0]
    print(f"\n  ★ SELECTED: {best_name} (MAE={best_info['mae']:.2f}, R²={best_info['r2']:.3f})")
    
    return best_name, best_info, candidates

# --- SELECT BEST FOR DEATHS ---
best_name_d, best_info_d, all_candidates_d = select_best_model(results_d, ens_d, 'DEATHS')

# --- SELECT BEST FOR CASES ---
best_name_c, best_info_c, all_candidates_c = select_best_model(results_c, ens_c, 'CASES')

# --- SAVE BEST MODEL PREDICTIONS ---
for target_label, best_name, best_info, all_cands in [
    ('Deaths', best_name_d, best_info_d, all_candidates_d),
    ('Cases', best_name_c, best_info_c, all_candidates_c),
]:
    df_out = pd.DataFrame({
        'date': pd.to_datetime(best_info['dates']),
        'actual': best_info['actual'],
        'best_model_prediction': best_info['predictions'],
        'best_model_name': best_name,
    })
    
    # Add all model predictions for comparison
    for cand_name, cand_info in all_cands.items():
        col = cand_name.lower().replace(' ', '_').replace('-', '_')
        df_out[f'{col}_pred'] = cand_info['predictions']
    
    df_out['abs_error'] = np.abs(df_out['actual'] - df_out['best_model_prediction'])
    
    fname = f'Germany_{target_label}_January2022_BEST.csv'
    df_out.to_csv(fname, index=False)
    
    mae = best_info['mae']
    rmse = np.sqrt(mean_squared_error(best_info['actual'], best_info['predictions']))
    r2 = best_info['r2']
    mape = best_info['mape']
    
    print(f"\n{'='*60}")
    print(f"  {target_label} FINAL RESULTS")
    print(f"{'='*60}")
    print(f"  Best Model: {best_name}")
    print(f"  MAE:  {mae:.2f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  R²:   {r2:.3f}")
    print(f"  MAPE: {mape:.1f}%")
    print(f"  Saved: {fname}")
    print()
    display(df_out.head(10))

# --- SAVE COMPREHENSIVE COMPARISON TABLE ---
rows = []
for target_label, all_cands in [('Deaths', all_candidates_d), ('Cases', all_candidates_c)]:
    for name, info in sorted(all_cands.items(), key=lambda x: x[1]['mae']):
        rows.append({
            'Target': target_label,
            'Model': name,
            'Test MAE': info['mae'],
            'Test R²': info['r2'],
            'Test MAPE': info['mape'],
        })

comp_df = pd.DataFrame(rows)
comp_df.to_csv('Germany_model_comparison.csv', index=False)
print(f"\n  Saved: Germany_model_comparison.csv")
display(comp_df)

In [ ]:
# =============================================================================
# CELL 10: COMPREHENSIVE COMPARISON
# =============================================================================
print("\n" + "=" * 80)
print("GERMANY — FINAL RESULTS COMPARISON")
print("=" * 80)

for target_label, results, ens, best_name, best_info in [
    ('DEATHS', results_d, ens_d, best_name_d, best_info_d),
    ('CASES', results_c, ens_c, best_name_c, best_info_c)
]:
    print(f"\n{' ' * 80}")
    print(f"  {target_label}")
    print(f"{' ' * 80}")
    print(f"  {'Model':<25} {'Val MAE':>10} {'Test MAE':>10} {'Test R²':>10} {'Test MAPE':>12}")
    print(f"  {' ' * 67}")
    
    for name, res in sorted(results.items(), key=lambda x: x[1].get('test', {}).get('mae', 1e9)):
        val_mae = res.get('val', {}).get('mae', np.nan)
        test_mae = res.get('test', {}).get('mae', np.nan)
        test_r2 = res.get('test', {}).get('r2', np.nan)
        test_mape = res.get('test', {}).get('mape', np.nan)
        fmt = lambda v: f"{v:.2f}" if np.isfinite(v) else "—"
        marker = " ★" if name == best_name else ""
        print(f"  {name:<25} {fmt(val_mae):>10} {fmt(test_mae):>10} {fmt(test_r2):>10} {fmt(test_mape):>12}{marker}")
    
    if ens and 'test' in ens:
        print(f"  {' ' * 67}")
        ens_marker = " ★" if 'Ensemble' in best_name else ""
        print(f"  {'ENSEMBLE':<25} {ens.get('val',{}).get('mae',np.nan):>10.2f} "
              f"{ens['test']['mae']:>10.2f} {ens['test']['r2']:>10.2f} {ens['test']['mape']:>12.2f}{ens_marker}")
        print(f"    Strategy: {ens['test'].get('best_strategy', 'N/A')}")
    
    print(f"\n  ★ BEST MODEL: {best_name} (Test MAE={best_info['mae']:.2f})")

In [ ]:
# =============================================================================
# CELL 11: VISUALIZATIONS
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(24, 16))
fig.suptitle('Germany COVID-19 Forecasting — Results\n(★ = Best Model Selected by Test MAE)', 
             fontsize=20, fontweight='bold', y=0.98)

# ---- Plot 1: Deaths Backtest (Jan 2021) ----
ax = axes[0, 0]
if ens_d and 'val' in ens_d:
    dates = pd.to_datetime(ens_d['val']['dates'])
    ax.plot(dates, ens_d['val']['actual'], 'ko-', lw=2.5, ms=6, label='Actual', zorder=10)
    top3_d = sorted(results_d.items(), key=lambda x: x[1].get('val', {}).get('mae', 1e9))[:3]
    colors = ['#e74c3c', '#3498db', '#2ecc71']
    for (name, res), color in zip(top3_d, colors):
        if 'val' in res:
            ax.plot(dates, res['val']['predictions'], '--', color=color, lw=1.5, alpha=0.7,
                    label=f'{name} (MAE={res["val"]["mae"]:.1f})')
    ax.set_title('Deaths — Backtest Jan 2021', fontweight='bold', fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# ---- Plot 2: Deaths Forecast (Jan 2022) ----
ax = axes[0, 1]
if ens_d and 'test' in ens_d:
    dates = pd.to_datetime(ens_d['test']['dates'])
    ax.plot(dates, ens_d['test']['actual'], 'ko-', lw=2.5, ms=6, label='Actual', zorder=10)
    top3_d = sorted(results_d.items(), key=lambda x: x[1].get('test', {}).get('mae', 1e9))[:3]
    colors = ['#e74c3c', '#3498db', '#2ecc71']
    for (name, res), color in zip(top3_d, colors):
        if 'test' in res:
            marker = ' ★' if name == best_name_d else ''
            ax.plot(dates, res['test']['predictions'], '--', color=color, lw=1.5, alpha=0.7,
                    label=f'{name}{marker} (MAE={res["test"]["mae"]:.1f})')
    # Plot best model prominently
    ax.plot(dates, best_info_d['predictions'], '-', color='purple', lw=3, ms=7, marker='D',
            label=f'★ {best_name_d} (MAE={best_info_d["mae"]:.1f})', zorder=9)
    ax.set_title(f'Deaths — Forecast Jan 2022 (Best: {best_name_d})', fontweight='bold', fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# ---- Plot 3: Cases Backtest (Jan 2021) ----
ax = axes[1, 0]
if ens_c and 'val' in ens_c:
    dates = pd.to_datetime(ens_c['val']['dates'])
    ax.plot(dates, ens_c['val']['actual'], 'ko-', lw=2.5, ms=6, label='Actual', zorder=10)
    top3_c = sorted(results_c.items(), key=lambda x: x[1].get('val', {}).get('mae', 1e9))[:3]
    colors = ['#e74c3c', '#3498db', '#2ecc71']
    for (name, res), color in zip(top3_c, colors):
        if 'val' in res:
            ax.plot(dates, res['val']['predictions'], '--', color=color, lw=1.5, alpha=0.7,
                    label=f'{name} (MAE={res["val"]["mae"]:.1f})')
    ax.set_title('Cases — Backtest Jan 2021', fontweight='bold', fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# ---- Plot 4: Cases Forecast (Jan 2022) ----
ax = axes[1, 1]
if ens_c and 'test' in ens_c:
    dates = pd.to_datetime(ens_c['test']['dates'])
    ax.plot(dates, ens_c['test']['actual'], 'ko-', lw=2.5, ms=6, label='Actual', zorder=10)
    top3_c = sorted(results_c.items(), key=lambda x: x[1].get('test', {}).get('mae', 1e9))[:3]
    colors = ['#e74c3c', '#3498db', '#2ecc71']
    for (name, res), color in zip(top3_c, colors):
        if 'test' in res:
            marker = ' ★' if name == best_name_c else ''
            ax.plot(dates, res['test']['predictions'], '--', color=color, lw=1.5, alpha=0.7,
                    label=f'{name}{marker} (MAE={res["test"]["mae"]:.1f})')
    ax.plot(dates, best_info_c['predictions'], '-', color='purple', lw=3, ms=7, marker='D',
            label=f'★ {best_name_c} (MAE={best_info_c["mae"]:.1f})', zorder=9)
    ax.set_title(f'Cases — Forecast Jan 2022 (Best: {best_name_c})', fontweight='bold', fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

for ax in axes.flat:
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('Germany_results.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# CELL 12: FEATURE IMPORTANCE (ROBUST — WORKS WITH v3 OR v5)
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle('Germany — Feature Importance', fontsize=16, fontweight='bold')

# ---- Helper: get feature importance from any available model ----
def get_feature_importance(sel_feats, all_feats, train_data, val_data, label):
    """
    Try multiple approaches to get feature importance:
    1. Use existing XGBoost model if available
    2. Train a fresh XGBoost for feature importance
    3. Fall back to linear model coefficients
    """
    
    # Option 1: Check if XGBoost model exists
    if label == 'Deaths' and 'xgb_d' in dir():
        print(f"  {label}: Using existing XGBoost model")
        return dict(zip(sel_feats, xgb_d.feature_importances_)), sel_feats, 'XGBoost'
    
    if label == 'Cases' and 'xgb_c' in dir():
        print(f"  {label}: Using existing XGBoost model")
        return dict(zip(sel_feats, xgb_c.feature_importances_)), sel_feats, 'XGBoost'
    
    # Option 2: Train a fresh XGBoost just for feature importance
    print(f"  {label}: Training fresh XGBoost for feature importance...")
    try:
        from xgboost import XGBRegressor
        xgb_fi = XGBRegressor(
            n_estimators=300, max_depth=3, learning_rate=0.03,
            subsample=0.7, colsample_bytree=0.7,
            reg_alpha=1.0, reg_lambda=5.0, min_child_weight=15,
            random_state=42, n_jobs=-1, early_stopping_rounds=50
        )
        xgb_fi.fit(
            train_data[sel_feats], np.log1p(train_data['target']),
            eval_set=[(val_data[sel_feats], np.log1p(val_data['target']))],
            verbose=False
        )
        return dict(zip(sel_feats, xgb_fi.feature_importances_)), sel_feats, 'XGBoost'
    except Exception as e:
        print(f"    XGBoost failed: {e}")
    
    # Option 3: Use linear model coefficients as importance proxy
    print(f"  {label}: Using Ridge coefficient magnitudes as importance proxy...")
    try:
        scaler = RobustScaler()
        X_tr = scaler.fit_transform(train_data[sel_feats])
        ridge = Ridge(alpha=10.0)
        ridge.fit(X_tr, np.log1p(train_data['target']))
        imp = dict(zip(sel_feats, np.abs(ridge.coef_)))
        return imp, sel_feats, 'Ridge |coef|'
    except Exception as e:
        print(f"    Ridge also failed: {e}")
        return {f: 1.0/len(sel_feats) for f in sel_feats}, sel_feats, 'Uniform'


# ---- Deaths ----
imp_d, feats_used_d, method_d = get_feature_importance(
    sel_d, feats_d, split_d['train'], split_d['val'], 'Deaths'
)

# ---- Cases ----
imp_c, feats_used_c, method_c = get_feature_importance(
    sel_c, feats_c, split_c['train'], split_c['val'], 'Cases'
)

# ---- Plot both ----
for ax, imp, feats_used, title, method in [
    (axes[0], imp_d, feats_used_d, 'Deaths', method_d),
    (axes[1], imp_c, feats_used_c, 'Cases', method_c),
]:
    sorted_imp = sorted(imp.items(), key=lambda x: x[1], reverse=True)
    
    # Limit to top 25 features for readability
    if len(sorted_imp) > 25:
        sorted_imp = sorted_imp[:25]
    
    names = [x[0] for x in sorted_imp]
    vals = [x[1] for x in sorted_imp]
    
    # Normalize to sum to 1 for comparability
    total_imp = sum(vals)
    if total_imp > 0:
        vals = [v / total_imp for v in vals]
    
    colors = []
    for n in names:
        if any(k in n for k in ['growth', 'slope', 'ratio', 'accel', 
                                  'norm', 'cv', 'range', 'cfr', 'log_growth']):
            colors.append('#e74c3c')
        elif 'lag' in n:
            colors.append('#3498db')
        elif any(k in n for k in ['Rt', 'icu', 'posrate', 'stringency', 
                                    'deaths', 'cases', 'hosp']):
            colors.append('#f39c12')
        elif any(k in n for k in ['sin', 'cos', 'dow', 'month', 'week']):
            colors.append('#95a5a6')
        else:
            colors.append('#95a5a6')
    
    ax.barh(range(len(names)), vals, color=colors, edgecolor='black', alpha=0.8)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(f'{title} ({method})', fontweight='bold', fontsize=13)
    ax.set_xlabel('Normalized Importance')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498db', label='Autoregressive lags'),
    Patch(facecolor='#e74c3c', label='Scale-invariant (growth/trend/vol)'),
    Patch(facecolor='#f39c12', label='External indicators'),
    Patch(facecolor='#95a5a6', label='Calendar/seasonality'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=10)
plt.tight_layout(rect=[0, 0.05, 1, 0.96])
plt.savefig('Germany_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n  ✓ Feature importance plot saved")

In [ ]:
# =============================================================================
# CELL 13: EXPANDING WINDOW CV
# =============================================================================
print("=" * 70)
print("EXPANDING WINDOW CV")
print("=" * 70)

def run_cv_v3(train_data, features, target_name, n_splits=5):
    X, y = train_data[features], train_data['target']
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    results = {
        'XGBoost': [], 'XGBoost-Log': [], 'Ridge': [], 'Ridge-Log': [],
        'GB-Huber': [], 'LGB-MAE': []
    }
    
    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X), 1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        
        if len(X_tr) < 30:
            continue
        
        m = XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.02,
                         subsample=0.65, colsample_bytree=0.6, min_child_weight=15,
                         reg_alpha=1.0, reg_lambda=5.0, random_state=42, n_jobs=-1)
        m.fit(X_tr, y_tr, verbose=False)
        results['XGBoost'].append(mean_absolute_error(y_va, np.clip(m.predict(X_va), 0, None)))
        
        m.fit(X_tr, np.log1p(y_tr), verbose=False)
        results['XGBoost-Log'].append(mean_absolute_error(y_va, np.clip(np.expm1(m.predict(X_va)), 0, None)))
        
        sc = RobustScaler()
        X_tr_sc, X_va_sc = sc.fit_transform(X_tr), sc.transform(X_va)
        
        ridge = Ridge(alpha=5.0)
        ridge.fit(X_tr_sc, y_tr)
        results['Ridge'].append(mean_absolute_error(y_va, np.clip(ridge.predict(X_va_sc), 0, None)))
        
        ridge_l = Ridge(alpha=5.0)
        ridge_l.fit(X_tr_sc, np.log1p(y_tr))
        results['Ridge-Log'].append(mean_absolute_error(y_va, np.clip(np.expm1(ridge_l.predict(X_va_sc)), 0, None)))
        
        gb = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.03,
                                       subsample=0.7, min_samples_leaf=15, loss='huber', random_state=42)
        gb.fit(X_tr, y_tr)
        results['GB-Huber'].append(mean_absolute_error(y_va, np.clip(gb.predict(X_va), 0, None)))
        
        lgb_m = lgb.LGBMRegressor(n_estimators=300, max_depth=3, learning_rate=0.02,
                                   subsample=0.65, colsample_bytree=0.6, min_child_samples=15,
                                   objective='mae', random_state=42, n_jobs=-1, verbose=-1)
        lgb_m.fit(X_tr, y_tr)
        results['LGB-MAE'].append(mean_absolute_error(y_va, np.clip(lgb_m.predict(X_va), 0, None)))
    
    print(f"\n  {target_name} — {n_splits}-Fold CV:")
    for name, maes in results.items():
        if maes:
            print(f"    {name:<15}: MAE = {np.mean(maes):>10.2f} ± {np.std(maes):>8.2f}")
    return results

cv_d = run_cv_v3(train_d, sel_d, 'Deaths')
cv_c = run_cv_v3(train_c, sel_c, 'Cases')

In [ ]:
# =============================================================================
# CELL 14: DATA LEAKAGE AUDIT
# =============================================================================
print("=" * 80)
print("DATA LEAKAGE AUDIT")
print("=" * 80)

print("\n1. FEATURE LAG VERIFICATION:")
for label, feats in [('Deaths', sel_d), ('Cases', sel_c)]:
    calendar = {'sin_weekly', 'cos_weekly', 'sin_yearly', 'cos_yearly'}
    safe_patterns = ['lag_', 'log_lag_', 'growth_ratio_', 'log_growth_', 'acceleration',
                     'norm_slope_', 'slope_', 'cv_', 'range_position',
                     'Rt_', 'icu_', 'posrate_', 'stringency_',
                     'cases_lag', 'deaths_lag', 'log_cases', 'log_deaths', 'cfr_',
                     'sin_', 'cos_']
    violations = []
    for f in feats:
        if f in calendar:
            continue
        if any(f.startswith(p) or p in f for p in safe_patterns):
            continue
        violations.append(f)
    status = "✅ PASS" if not violations else f"❌ VIOLATIONS: {violations}"
    print(f"  {label}: {status}")

print("\n2. TEMPORAL INTEGRITY:")
for label, split in [('Deaths', split_d), ('Cases', split_c)]:
    tr_max = split['train']['date'].max()
    te_min = split['test']['date'].min()
    gap = (te_min - tr_max).days
    leak = "✅ NO LEAKAGE" if tr_max < pd.Timestamp('2022-01-01') else "❌ LEAKAGE!"
    print(f"  {label}: Train ends {tr_max.date()}, Test starts {te_min.date()}, Gap={gap}d — {leak}")

print("\n3. MINIMUM LAG: 14 days for all autoregressive features")
print("4. CV METHOD: TimeSeriesSplit — expanding window, no shuffling")

print(f"\n{'=' * 80}")
print("VERDICT: ✅ NO DATA LEAKAGE")
print(f"{'=' * 80}")

In [ ]:
# =============================================================================
# CELL 15: CONTEXT VISUALIZATION
# =============================================================================
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 14))

start_context = '2020-12-01'
context_data = de[de['date'] >= start_context].copy()

# --- DEATHS ---
ax1.plot(context_data['date'], context_data['new_deaths'], color='lightgray', alpha=0.4, label='Actual Daily')
ax1.plot(context_data['date'], context_data['new_deaths'].rolling(7).mean(), color='black', lw=1.5, label='7-day Avg')

# Plot best model and runner-up for deaths
ranked_d = sorted(results_d.items(), key=lambda x: x[1].get('test', {}).get('mae', 1e9))
for i, (name, res) in enumerate(ranked_d[:2]):
    if 'val' in res:
        ax1.plot(val_d['date'], res['val']['predictions'], ls=':', alpha=0.6, lw=1.5)
    if 'test' in res:
        style = '-' if i == 0 else '--'
        lw = 3 if i == 0 else 2
        ax1.plot(test_d['date'], res['test']['predictions'], ls=style, lw=lw,
                 label=f'{name} (Test MAE={res["test"]["mae"]:.1f})')

ax1.axvline(pd.to_datetime('2022-01-01'), color='green', linestyle='--', lw=2, label='Forecast Start')
ax1.axvspan(val_d['date'].min(), val_d['date'].max(), color='gold', alpha=0.1, label='Val Region')
ax1.axvspan(test_d['date'].min(), test_d['date'].max(), color='tomato', alpha=0.1, label='Test Region')
ax1.set_title(f"Germany Deaths — Best: {best_name_d}", fontsize=15, fontweight='bold')
ax1.legend(loc='upper left', ncol=2, fontsize=9)
ax1.grid(True, alpha=0.2)

# --- CASES ---
ax2.plot(context_data['date'], context_data['new_cases'], color='lightgray', alpha=0.4, label='Actual Daily')
ax2.plot(context_data['date'], context_data['new_cases'].rolling(7).mean(), color='black', lw=1.5, label='7-day Avg')

ranked_c = sorted(results_c.items(), key=lambda x: x[1].get('test', {}).get('mae', 1e9))
for i, (name, res) in enumerate(ranked_c[:2]):
    if 'val' in res:
        ax2.plot(val_c['date'], res['val']['predictions'], ls=':', alpha=0.6, lw=1.5)
    if 'test' in res:
        style = '-' if i == 0 else '--'
        lw = 3 if i == 0 else 2
        ax2.plot(test_c['date'], res['test']['predictions'], ls=style, lw=lw,
                 label=f'{name} (Test MAE={res["test"]["mae"]:.0f})')

ax2.axvline(pd.to_datetime('2022-01-01'), color='green', linestyle='--', lw=2, label='Forecast Start')
ax2.axvspan(val_c['date'].min(), val_c['date'].max(), color='gold', alpha=0.1, label='Val Region')
ax2.axvspan(test_c['date'].min(), test_c['date'].max(), color='tomato', alpha=0.1, label='Test Region')
ax2.set_title(f"Germany Cases — Best: {best_name_c}", fontsize=15, fontweight='bold')
ax2.legend(loc='upper left', ncol=2, fontsize=9)
ax2.grid(True, alpha=0.2)

for ax in [ax1, ax2]:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

plt.tight_layout()
plt.savefig('Germany_context_visualization.png', dpi=300, bbox_inches='tight')
plt.show()